# Calgary Traffic Risk Intelligence

## 05 — Predictive Modeling

This notebook builds an interpretable forecasting model for **hourly recorded
traffic-incident activity across Calgary**.

The target is the number of recorded incidents during each observed hour.

### Objectives

- Load the complete hourly incident + weather dataset
- Use full calendar years only
- Build temporal and weather-based model features
- Use a chronological train/test split
- Create a strong historical-pattern baseline
- Train a count-aware gradient-boosting model
- Compare model performance against the baseline
- Inspect prediction errors
- Estimate permutation feature importance
- Save the trained pipeline and supporting artifacts
- Create a reusable prediction function for the Streamlit app

> This model estimates citywide hourly incident activity from historical patterns.
> It does not predict whether a specific driver will crash and should not be used
> as a real-time navigation or safety system.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline

# Display tables clearly.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)

## Load Modelling Data

This notebook expects `04_weather_and_risk_analysis.ipynb` to have produced the
hourly exposure dataset.

In [2]:
# Find the project root.
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HOURLY_DATA_FILE = (
    CLEAN_DIR / "hourly_incident_weather.csv"
)

print("Project root:", PROJECT_ROOT)
print("Hourly modelling data exists:", HOURLY_DATA_FILE.exists())

if not HOURLY_DATA_FILE.exists():
    raise FileNotFoundError(
        "Run 04_weather_and_risk_analysis.ipynb first."
    )

Project root: c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence
Hourly modelling data exists: True


In [3]:
# Load the hourly exposure dataset.
hourly = pd.read_csv(
    HOURLY_DATA_FILE,
    low_memory=False,
)

required_columns = {
    "weather_hour_utc",
    "weather_hour_local",
    "year",
    "month",
    "weekday_number",
    "weekday",
    "hour",
    "incident_count",
    "inferred_collection_gap",
    "temp_c",
    "rel_hum_pct",
    "wind_speed_kmh",
    "visibility_km",
    "freezing_temp",
    "low_visibility",
    "snow_flag",
    "rain_flag",
    "adverse_weather_flag",
}

missing_columns = sorted(
    required_columns - set(hourly.columns)
)

if missing_columns:
    raise KeyError(
        "The hourly dataset is missing columns from corrected Notebook 04: "
        f"{missing_columns}"
    )

hourly["weather_hour_utc"] = pd.to_datetime(
    hourly["weather_hour_utc"],
    utc=True,
)

hourly["weather_hour_local"] = pd.to_datetime(
    hourly["weather_hour_local"],
    utc=True,
).dt.tz_convert("America/Edmonton")

print(
    f"Hourly rows: {hourly.shape[0]:,} × "
    f"{hourly.shape[1]} columns"
)

display(hourly.head())

Hourly rows: 86,184 × 27 columns


,weather_hour_utc,temp_c,dew_point_c,rel_hum_pct,precip_mm,wind_speed_kmh,visibility_km,station_pressure_kpa,weather_text,weather_hour_local,snow_flag,rain_flag,freezing_temp,extreme_cold,low_visibility,high_wind,measurable_precip,adverse_weather_flag,incident_hour_utc,incident_count,year,month,weekday_number,weekday,hour,weekend,inferred_collection_gap
0,2016-12-01 07:00:00+00:00,-4.8,-6.1,91.0,NaN,0.0,6.4,88.32,Snow,2016-12-01 00:00:00-07:00,True,False,True,False,False,False,NaN,True,NaN,0,2016,12,3,Thursday,0,False,False
1,2016-12-01 08:00:00+00:00,-4.8,-5.9,92.0,NaN,0.0,6.4,88.34,Snow,2016-12-01 01:00:00-07:00,True,False,True,False,False,False,NaN,True,NaN,0,2016,12,3,Thursday,1,False,False
2,2016-12-01 09:00:00+00:00,-4.4,-5.5,92.0,NaN,0.0,3.2,88.39,Snow,2016-12-01 02:00:00-07:00,True,False,True,False,True,False,NaN,True,NaN,0,2016,12,3,Thursday,2,False,False
3,2016-12-01 10:00:00+00:00,-3.7,-4.7,93.0,NaN,5.0,4.0,88.40,Snow,2016-12-01 03:00:00-07:00,True,False,True,False,True,False,NaN,True,NaN,0,2016,12,3,Thursday,3,False,False
4,2016-12-01 11:00:00+00:00,-3.5,-4.8,91.0,NaN,5.0,4.8,88.43,Snow,2016-12-01 04:00:00-07:00,True,False,True,False,True,False,NaN,True,NaN,0,2016,12,3,Thursday,4,False,False


## Modelling Window

The incident archive contains documented collection gaps, so calendar boundaries alone are not enough to define valid modelling exposure.

The model uses:

- **Training calendar window:** 2017–2024
- **Testing calendar window:** 2025
- **Coverage rule:** exclude hours marked by Notebook 04 as inferred collection gaps

The test observations are later in time than every training observation, which better represents forecasting than a random train/test split.

Because the ECCC hourly precipitation field is unavailable in this project snapshot, `precip_mm` is not used as a model feature.

In [4]:
# Restrict modelling to the chronological calendar window
# and exclude hours inside inferred source-collection gaps.
def to_boolean(series):
    """Parse common CSV boolean representations without treating 'False' as truthy text."""

    if pd.api.types.is_bool_dtype(series.dtype):
        return series.astype("boolean")

    if pd.api.types.is_numeric_dtype(series.dtype):
        return series.map(
            {1: True, 0: False}
        ).astype("boolean")

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return normalized.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    ).astype("boolean")


hourly["inferred_collection_gap"] = to_boolean(
    hourly["inferred_collection_gap"]
).fillna(False)

calendar_window = hourly[
    hourly["year"].between(
        2017,
        2025,
    )
].copy()

model_data = calendar_window[
    ~calendar_window[
        "inferred_collection_gap"
    ]
].copy()

train_data = model_data[
    model_data["year"].between(
        2017,
        2024,
    )
].copy()

test_data = model_data[
    model_data["year"] == 2025
].copy()

if train_data.empty or test_data.empty:
    raise ValueError(
        "Chronological train/test data is empty. Rerun corrected Notebook 04."
    )

print(
    "Training period:",
    train_data["weather_hour_local"].min(),
    "to",
    train_data["weather_hour_local"].max(),
)

print(
    "Testing period:",
    test_data["weather_hour_local"].min(),
    "to",
    test_data["weather_hour_local"].max(),
)

print("Training rows:", f"{len(train_data):,}")
print("Testing rows:", f"{len(test_data):,}")

print(
    "Excluded inferred-gap hours:",
    f"{calendar_window['inferred_collection_gap'].sum():,}",
)

Training period: 2017-02-08 09:00:00-07:00 to 2024-12-31 23:00:00-07:00
Testing period: 2025-01-01 00:00:00-07:00 to 2025-12-31 23:00:00-07:00
Training rows: 66,587
Testing rows: 7,681
Excluded inferred-gap hours: 4,620


In [5]:
# Check the target distribution.
target_summary = (
    model_data["incident_count"]
    .describe()
    .to_frame("incident_count")
)

display(target_summary)

print(
    "Zero-incident hours:",
    f"{(model_data['incident_count'] == 0).mean() * 100:.2f}%"
)

,incident_count
count,74268.000000
mean,0.787998
std,1.199307
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,19.000000


Zero-incident hours: 56.29%


## Feature Engineering

Cyclical sine/cosine features preserve the circular structure of:

- hour of day
- weekday
- month

For example, hour 23 and hour 0 should be considered close in time.

In [6]:
def add_model_features(df):
    """Create numeric temporal and weather features for modelling."""

    result = df.copy()

    # Cyclical time features.
    result["hour_sin"] = np.sin(
        2 * np.pi * result["hour"] / 24
    )
    result["hour_cos"] = np.cos(
        2 * np.pi * result["hour"] / 24
    )

    result["dow_sin"] = np.sin(
        2 * np.pi * result["weekday_number"] / 7
    )
    result["dow_cos"] = np.cos(
        2 * np.pi * result["weekday_number"] / 7
    )

    result["month_sin"] = np.sin(
        2 * np.pi * (result["month"] - 1) / 12
    )
    result["month_cos"] = np.cos(
        2 * np.pi * (result["month"] - 1) / 12
    )

    # Simple interpretable time flags.
    result["weekend_feature"] = (
        result["weekday_number"] >= 5
    ).astype(int)

    result["rush_hour"] = (
        result["hour"].between(6, 9)
        | result["hour"].between(15, 18)
    ).astype(int)

    # Parse nullable weather-condition flags safely. Unknown values remain NaN
    # and are handled by the model pipeline rather than silently becoming False.
    flag_columns = [
        "freezing_temp",
        "low_visibility",
        "snow_flag",
        "rain_flag",
        "adverse_weather_flag",
    ]

    for column in flag_columns:
        parsed = to_boolean(
            result[column]
        )

        result[column] = parsed.map(
            {
                True: 1.0,
                False: 0.0,
            }
        ).astype(float)

    return result

In [7]:
# Engineer features independently for train and test data.
train_data = add_model_features(train_data)
test_data = add_model_features(test_data)

FEATURE_COLUMNS = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "weekend_feature",
    "rush_hour",
    "temp_c",
    "rel_hum_pct",
    "wind_speed_kmh",
    "visibility_km",
    "freezing_temp",
    "low_visibility",
    "snow_flag",
    "rain_flag",
    "adverse_weather_flag",
]

TARGET_COLUMN = "incident_count"

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN].astype(float)

X_test = test_data[FEATURE_COLUMNS]
y_test = test_data[TARGET_COLUMN].astype(float)

print("Features:", len(FEATURE_COLUMNS))
print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)

Features: 17
Training matrix: (66587, 17)
Testing matrix: (7681, 17)


In [8]:
# Inspect modelling-feature missingness.
feature_missing = (
    X_train.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("training_missing_percent")
)

display(feature_missing)

assert "precip_mm" not in FEATURE_COLUMNS

print(
    "Scikit-learn version:",
    sklearn.__version__,
)

,training_missing_percent
temp_c,0.11
wind_speed_kmh,0.11
freezing_temp,0.11
rel_hum_pct,0.11
visibility_km,0.10
low_visibility,0.10
hour_sin,0.00
dow_cos,0.00
hour_cos,0.00
rush_hour,0.00


Scikit-learn version: 1.9.1


## Historical-Pattern Baseline

A useful machine-learning model should outperform a simple alternative.

The baseline predicts the historical training mean for each combination of:

- weekday
- hour of day

If a combination is missing, the overall training mean is used.

In [9]:
# Mean incident count by weekday and hour in the training period.
baseline_lookup = (
    train_data
    .groupby(
        ["weekday_number", "hour"]
    )["incident_count"]
    .mean()
)

overall_train_mean = (
    train_data["incident_count"]
    .mean()
)

def baseline_predict(df):
    predictions = []

    for row in df[
        ["weekday_number", "hour"]
    ].itertuples(index=False):
        prediction = baseline_lookup.get(
            (
                row.weekday_number,
                row.hour,
            ),
            overall_train_mean,
        )

        predictions.append(prediction)

    return np.asarray(predictions)

baseline_predictions = baseline_predict(
    test_data
)

print(
    "Baseline prediction mean:",
    baseline_predictions.mean(),
)

Baseline prediction mean: 0.7746079075929461


## Model

The project uses `HistGradientBoostingRegressor` with a **Poisson loss** because:

- the target is a non-negative hourly count
- relationships between time/weather and incidents can be nonlinear
- histogram gradient boosting is efficient for this dataset size
- the final pipeline remains small enough for Streamlit deployment

A median imputer is included so occasional missing weather measurements do not
cause prediction failure.

In [10]:
# Build a reproducible count-aware regression pipeline.
# Missing numeric values are median-imputed and missingness indicators are added,
# allowing the model to distinguish imputed weather observations from measured ones.
model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "regressor",
            HistGradientBoostingRegressor(
                loss="poisson",
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=31,
                min_samples_leaf=30,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",True
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible.

In [11]:
# Train only on historical years through 2024.
model.fit(
    X_train,
    y_train,
)

print("Model training complete.")

Model training complete.


In [12]:
# Predict 2025 hourly incident counts.
model_predictions = model.predict(
    X_test
)

# Numerical safeguard for count metrics and display.
model_predictions = np.clip(
    model_predictions,
    0,
    None,
)

print(
    "Prediction range:",
    model_predictions.min(),
    "to",
    model_predictions.max(),
)

Prediction range: 0.09783081391112294 to 4.7300943453561075


## Model Evaluation

In [13]:
def regression_metrics(y_true, y_pred):
    """Return the main project regression metrics."""

    y_pred_nonzero = np.clip(
        y_pred,
        1e-6,
        None,
    )

    return {
        "MAE": mean_absolute_error(
            y_true,
            y_pred,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        ),
        "R2": r2_score(
            y_true,
            y_pred,
        ),
        "Mean Poisson Deviance":
            mean_poisson_deviance(
                y_true,
                y_pred_nonzero,
            ),
    }


baseline_metrics = regression_metrics(
    y_test,
    baseline_predictions,
)

model_metrics = regression_metrics(
    y_test,
    model_predictions,
)

metrics_table = pd.DataFrame(
    [
        {
            "model": "Weekday-hour baseline",
            **baseline_metrics,
        },
        {
            "model": "Poisson HistGradientBoosting",
            **model_metrics,
        },
    ]
)

display(
    metrics_table.round(4)
)

,model,MAE,RMSE,R2,Mean Poisson Deviance
0,Weekday-hour baseline,0.8026,1.1976,0.1619,1.3861
1,Poisson HistGradientBoosting,0.7970,1.1669,0.2044,1.3329


In [14]:
# Calculate relative MAE improvement over the baseline.
baseline_mae = baseline_metrics["MAE"]
model_mae = model_metrics["MAE"]

mae_improvement_percent = (
    (baseline_mae - model_mae)
    / baseline_mae
    * 100
)

print(
    "MAE improvement over baseline:",
    f"{mae_improvement_percent:.2f}%"
)

MAE improvement over baseline: 0.70%


A negative improvement would mean the ML model did not beat the historical
weekday/hour baseline. That result should be reported honestly rather than hidden.

## Test-Period Predictions

In [15]:
# Store held-out 2025 actual and predicted values.
test_predictions = test_data[
    [
        "weather_hour_utc",
        "weather_hour_local",
        "year",
        "month",
        "weekday_number",
        "weekday",
        "hour",
        "incident_count",
    ]
].copy()

test_predictions[
    "baseline_prediction"
] = baseline_predictions

test_predictions[
    "model_prediction"
] = model_predictions

test_predictions[
    "absolute_error"
] = np.abs(
    test_predictions["incident_count"]
    - test_predictions["model_prediction"]
)

display(test_predictions.head())

,weather_hour_utc,weather_hour_local,year,month,weekday_number,weekday,hour,incident_count,baseline_prediction,model_prediction,absolute_error
70872,2025-01-01 07:00:00+00:00,2025-01-01 00:00:00-07:00,2025,1,2,Wednesday,0,4,0.204030,0.392838,3.607162
70873,2025-01-01 08:00:00+00:00,2025-01-01 01:00:00-07:00,2025,1,2,Wednesday,1,4,0.130982,0.144521,3.855479
70874,2025-01-01 09:00:00+00:00,2025-01-01 02:00:00-07:00,2025,1,2,Wednesday,2,1,0.156171,0.125299,0.874701
70875,2025-01-01 10:00:00+00:00,2025-01-01 03:00:00-07:00,2025,1,2,Wednesday,3,1,0.070529,0.137641,0.862359
70876,2025-01-01 11:00:00+00:00,2025-01-01 04:00:00-07:00,2025,1,2,Wednesday,4,0,0.146096,0.217556,0.217556


In [16]:
# Compare monthly actual totals and model-estimated totals.
# Months fully inside a collection gap will be absent instead of appearing as zero.
monthly_test = (
    test_predictions
    .groupby("month")
    .agg(
        observed_hours=(
            "incident_count",
            "size",
        ),
        actual_incidents=(
            "incident_count",
            "sum",
        ),
        predicted_incidents=(
            "model_prediction",
            "sum",
        ),
        baseline_incidents=(
            "baseline_prediction",
            "sum",
        ),
    )
    .reset_index()
)

display(monthly_test)

,month,observed_hours,actual_incidents,predicted_incidents,baseline_incidents
0,1,692,598,535.455651,551.515660
1,2,672,814,643.951319,519.693074
2,3,743,530,469.687323,564.752847
3,4,720,547,419.514525,562.066465
4,5,471,366,287.803813,357.946040
5,7,710,634,527.708843,555.591830
6,8,744,590,560.256340,568.407933
7,9,720,603,623.348398,558.998058
8,10,744,656,633.633102,583.359600
9,11,721,819,602.455234,547.017042


In [17]:
# Plot monthly actual vs predicted incident totals.
monthly_long = monthly_test.melt(
    id_vars="month",
    value_vars=[
        "actual_incidents",
        "predicted_incidents",
        "baseline_incidents",
    ],
    var_name="series",
    value_name="incident_count",
)

fig = px.line(
    monthly_long,
    x="month",
    y="incident_count",
    color="series",
    markers=True,
    title="2025 Monthly Actual vs Predicted Recorded Incidents",
    labels={
        "month": "Month",
        "incident_count": "Recorded / estimated incidents",
        "series": "Series",
    },
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

## Error Analysis

In [18]:
# Mean absolute error by hour of day.
error_by_hour = (
    test_predictions
    .groupby("hour")
    .agg(
        observations=("absolute_error", "size"),
        mean_absolute_error=("absolute_error", "mean"),
        actual_mean=("incident_count", "mean"),
        predicted_mean=("model_prediction", "mean"),
    )
    .reset_index()
)

display(error_by_hour)

,hour,observations,mean_absolute_error,actual_mean,predicted_mean
0,0,320,0.472147,0.353125,0.277078
1,1,321,0.525783,0.442368,0.213187
2,2,319,0.401546,0.304075,0.168346
3,3,320,0.338554,0.240625,0.159311
4,4,319,0.337292,0.225705,0.165356
5,5,319,0.447942,0.357367,0.189294
6,6,319,0.762126,0.746082,0.549543
7,7,319,0.997532,1.275862,1.065630
8,8,320,1.104574,1.431250,1.092046
9,9,320,0.913663,1.181250,0.964844


In [19]:
fig = px.line(
    error_by_hour,
    x="hour",
    y="mean_absolute_error",
    markers=True,
    title="2025 Model Error by Hour of Day",
    labels={
        "hour": "Hour",
        "mean_absolute_error": "Mean absolute error",
    },
)

fig.update_xaxes(
    dtick=1,
)

fig.show()

## Permutation Feature Importance

Permutation importance measures how much test-set prediction quality degrades
when one feature is randomly shuffled.

This provides model-level explainability without claiming that an important
feature is causal.

In [20]:
# Use a fixed test sample to keep the calculation reasonably fast.
importance_sample_size = min(
    10000,
    len(X_test),
)

importance_indices = (
    X_test
    .sample(
        n=importance_sample_size,
        random_state=42,
    )
    .index
)

X_importance = X_test.loc[
    importance_indices
]

y_importance = y_test.loc[
    importance_indices
]

importance_result = permutation_importance(
    model,
    X_importance,
    y_importance,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

feature_importance = pd.DataFrame(
    {
        "feature": FEATURE_COLUMNS,
        "importance_mean": importance_result.importances_mean,
        "importance_std": importance_result.importances_std,
    }
).sort_values(
    "importance_mean",
    ascending=False,
)

display(feature_importance)

,feature,importance_mean,importance_std
1,hour_cos,0.122308,0.002946
0,hour_sin,0.038957,0.002720
8,temp_c,0.025357,0.003030
2,dow_sin,0.025018,0.003335
4,month_sin,0.003443,0.001113
5,month_cos,0.003377,0.001120
11,visibility_km,0.002607,0.000722
6,weekend_feature,0.002146,0.001073
14,snow_flag,0.001990,0.000380
7,rush_hour,0.001889,0.000600


In [21]:
# Plot the most influential model features.
top_importance = (
    feature_importance
    .head(15)
    .sort_values("importance_mean")
)

fig = px.bar(
    top_importance,
    x="importance_mean",
    y="feature",
    orientation="h",
    error_x="importance_std",
    title="Permutation Feature Importance",
    labels={
        "importance_mean":
            "Increase in MAE after permutation",
        "feature":
            "Feature",
    },
)

fig.show()

## Reusable Prediction Helper

The function below prepares a single hypothetical hour in the same feature format
used during training.

It is intended for the final Streamlit dashboard.

In [22]:
def build_prediction_row(
    hour,
    weekday_number,
    month,
    temp_c,
    rel_hum_pct,
    wind_speed_kmh,
    visibility_km,
    snow_flag=False,
    rain_flag=False,
):
    """Create one model-ready hypothetical hourly scenario."""

    freezing_temp = int(
        pd.notna(temp_c)
        and temp_c <= 0
    )

    low_visibility = int(
        pd.notna(visibility_km)
        and visibility_km < 5
    )

    adverse_weather_flag = int(
        bool(snow_flag)
        or bool(rain_flag)
        or freezing_temp
        or low_visibility
        or (
            pd.notna(wind_speed_kmh)
            and wind_speed_kmh >= 50
        )
    )

    row = pd.DataFrame(
        [
            {
                "hour_sin":
                    np.sin(
                        2 * np.pi
                        * hour / 24
                    ),
                "hour_cos":
                    np.cos(
                        2 * np.pi
                        * hour / 24
                    ),
                "dow_sin":
                    np.sin(
                        2 * np.pi
                        * weekday_number / 7
                    ),
                "dow_cos":
                    np.cos(
                        2 * np.pi
                        * weekday_number / 7
                    ),
                "month_sin":
                    np.sin(
                        2 * np.pi
                        * (month - 1) / 12
                    ),
                "month_cos":
                    np.cos(
                        2 * np.pi
                        * (month - 1) / 12
                    ),
                "weekend_feature":
                    int(
                        weekday_number >= 5
                    ),
                "rush_hour":
                    int(
                        6 <= hour <= 9
                        or 15 <= hour <= 18
                    ),
                "temp_c": temp_c,
                "rel_hum_pct": rel_hum_pct,
                "wind_speed_kmh":
                    wind_speed_kmh,
                "visibility_km":
                    visibility_km,
                "freezing_temp":
                    freezing_temp,
                "low_visibility":
                    low_visibility,
                "snow_flag":
                    int(bool(snow_flag)),
                "rain_flag":
                    int(bool(rain_flag)),
                "adverse_weather_flag":
                    adverse_weather_flag,
            }
        ]
    )

    return row[FEATURE_COLUMNS]

In [23]:
def predict_hourly_incident_activity(**scenario):
    """Estimate citywide recorded incident count for one hour."""

    row = build_prediction_row(
        **scenario
    )

    prediction = float(
        model.predict(row)[0]
    )

    return max(
        0.0,
        prediction,
    )

In [24]:
# Example scenarios for a quick model sanity check.
example_scenarios = [
    {
        "name":
            "Weekday afternoon, mild and clear",
        "hour": 16,
        "weekday_number": 2,
        "month": 6,
        "temp_c": 18,
        "rel_hum_pct": 45,
        "wind_speed_kmh": 15,
        "visibility_km": 24,
        "snow_flag": False,
        "rain_flag": False,
    },
    {
        "name":
            "Weekday morning, freezing with snow",
        "hour": 8,
        "weekday_number": 1,
        "month": 1,
        "temp_c": -15,
        "rel_hum_pct": 85,
        "wind_speed_kmh": 25,
        "visibility_km": 3,
        "snow_flag": True,
        "rain_flag": False,
    },
]

scenario_results = []

for scenario in example_scenarios:
    model_inputs = {
        key: value
        for key, value in scenario.items()
        if key != "name"
    }

    scenario_results.append(
        {
            "scenario":
                scenario["name"],
            "estimated_incidents_in_hour":
                predict_hourly_incident_activity(
                    **model_inputs
                ),
        }
    )

display(
    pd.DataFrame(
        scenario_results
    )
)

,scenario,estimated_incidents_in_hour
0,"Weekday afternoon, mild and clear",2.35642
1,"Weekday morning, freezing with snow",1.74476


The scenario estimates are model outputs, not safety guarantees. They are useful
for demonstrating how temporal and weather inputs change the model's expected
citywide incident activity.

## Save Model and Supporting Artifacts

In [25]:
# File locations used later by the Streamlit application.
MODEL_FILE = (
    MODELS_DIR / "hourly_incident_model.joblib"
)

MODEL_METADATA_FILE = (
    MODELS_DIR / "hourly_incident_model_metadata.json"
)

MODEL_METRICS_FILE = (
    CLEAN_DIR / "model_metrics.csv"
)

FEATURE_IMPORTANCE_FILE = (
    CLEAN_DIR / "model_feature_importance.csv"
)

TEST_PREDICTIONS_FILE = (
    CLEAN_DIR / "model_test_predictions_2025.csv"
)

# Save trained pipeline.
joblib.dump(
    model,
    MODEL_FILE,
)

metrics_table.to_csv(
    MODEL_METRICS_FILE,
    index=False,
)

feature_importance.to_csv(
    FEATURE_IMPORTANCE_FILE,
    index=False,
)

test_predictions.to_csv(
    TEST_PREDICTIONS_FILE,
    index=False,
)

In [26]:
# Save metadata required to reproduce app-side predictions.
model_metadata = {
    "model_name":
        "Poisson HistGradientBoostingRegressor",
    "target":
        "hourly recorded Calgary traffic incident count",
    "training_years": [
        2017,
        2018,
        2019,
        2020,
        2021,
        2022,
        2023,
        2024,
    ],
    "test_year": 2025,
    "training_rows": int(len(train_data)),
    "test_rows": int(len(test_data)),
    "excluded_gap_hours": int(
        calendar_window["inferred_collection_gap"].sum()
    ),
    "feature_columns": FEATURE_COLUMNS,
    "baseline":
        "Historical mean by weekday_number and hour",
    "baseline_mae": float(
        baseline_metrics["MAE"]
    ),
    "model_mae": float(
        model_metrics["MAE"]
    ),
    "mae_improvement_percent": float(
        mae_improvement_percent
    ),
    "scikit_learn_version":
        sklearn.__version__,
    "imputer":
        "Median imputation with missing-value indicators",
    "collection_gap_rule":
        "Exclude hours inside >48-hour gaps between consecutive recorded incidents.",
    "limitations": [
        (
            "Predicts citywide hourly recorded incident "
            "activity, not individual crashes."
        ),
        (
            "Airport weather is used as a citywide "
            "weather proxy."
        ),
        (
            "Traffic-volume exposure is not included."
        ),
        (
            "Model relationships are predictive "
            "associations, not causal effects."
        ),
        (
            "The traffic-incidents archive can contain "
            "collection gaps; a conservative >48-hour "
            "gap heuristic is used to exclude obvious outages."
        ),
    ],
}

MODEL_METADATA_FILE.write_text(
    json.dumps(
        model_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:")
print(MODEL_FILE)
print(MODEL_METADATA_FILE)
print(MODEL_METRICS_FILE)
print(FEATURE_IMPORTANCE_FILE)
print(TEST_PREDICTIONS_FILE)

Saved:
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\models\hourly_incident_model.joblib
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\models\hourly_incident_model_metadata.json
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\model_metrics.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\model_feature_importance.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\model_test_predictions_2025.csv


## Final Validation

In [27]:
# Validate final model artifacts and predictions.
assert MODEL_FILE.exists()
assert MODEL_METADATA_FILE.exists()

assert len(model_predictions) == len(y_test)
assert len(test_predictions) == len(test_data)

assert np.isfinite(model_predictions).all()
assert (model_predictions >= 0).all()

assert set(FEATURE_COLUMNS) == set(
    X_train.columns
)

assert not train_data["inferred_collection_gap"].any()
assert not test_data["inferred_collection_gap"].any()

print("Predictive-model validation passed.")

Predictive-model validation passed.


## Predictive Modeling Summary

This stage completes the project's predictive component.

### Completed

- uses an hourly modelling frame containing genuine zero-incident hours
- excludes hours inside inferred source-collection gaps
- uses 2017–2024 as the chronological training window
- uses eligible 2025 hours as the held-out test window
- engineers cyclical time features
- incorporates temperature, humidity, wind, visibility, and weather-condition flags
- excludes the unavailable hourly precipitation field from predictive inputs
- builds a historical weekday/hour baseline
- trains a Poisson histogram gradient-boosting model
- compares MAE, RMSE, R², and Poisson deviance
- calculates improvement relative to the baseline
- estimates permutation feature importance
- creates reusable scenario-prediction functions
- saves the trained model and metadata, including the scikit-learn version

### Interpretation

The model estimates **citywide recorded traffic-incident activity per eligible observed hour**.

It does not predict a specific crash, determine whether a road is safe, or establish that any weather variable causes traffic incidents.

The traffic-incidents source can contain archive gaps, so the model evaluation should be described as a chronological held-out evaluation over observed 2025 coverage rather than a perfectly complete year.